# **Full Video Inference Pipeline**
---

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from detection_system.loader import load_model
from detection_system.inference import detect_video
from detection_system.utils import load_config, save_figure

cfg = load_config("config.yaml")
plt.style.use(cfg["figures"]["style"])
CONF_THRESHOLD = cfg["model"]["confidence_threshold"]
print("Setup complete")

Setup complete


In [ ]:
# ── Selecting model and video ────────────────────────────────────────────
model = load_model("small", models_dir=PROJECT_ROOT / "models")

real_video = PROJECT_ROOT / "data" / "raw" / "sample_traffic.mp4"
mock_video = PROJECT_ROOT / "data" / "mock" / "mock_video.mp4"
VIDEO_PATH = real_video if real_video.exists() else mock_video

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = OUTPUT_DIR / "annotated_output.mp4"
OUTPUT_JSON = OUTPUT_DIR / "detections.json"

print(f"Input  : {VIDEO_PATH}")
print(f"Output : {OUTPUT_VIDEO}")

Weights will be saved to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models
Model loaded: YOLOv8 SMALL | Classes: 80
Input  : c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\data\raw\sample_traffic.mp4
Output : c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\data\processed\annotated_output.mp4


In [ ]:
# ── Running detection on the full video ───────────────────────────────────

print("Running full video inference...")
print("(This may take several minutes on CPU for a long video)")
print()

results = detect_video(
    model=model,
    video_path=VIDEO_PATH,
    output_path=OUTPUT_VIDEO,
    confidence=CONF_THRESHOLD,
    max_frames=None,  # Set to 50 for a quick test
    verbose=True,
)

print(f"\n✓ Video inference complete")
print(f"  Frames processed : {results['frame_count']}")
print(f"  Output video     : {OUTPUT_VIDEO}")

Running full video inference...
(This may take several minutes on CPU for a long video)

Video: sample_traffic.mp4
  Resolution : 1920×1080
  FPS        : 25.0
  Frames     : 1501
  Output     : c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\data\processed\annotated_output.mp4

0: 384x640 7 cars, 2 trucks, 238.4ms
Speed: 5.9ms preprocess, 238.4ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8 cars, 2 trucks, 188.1ms
Speed: 4.7ms preprocess, 188.1ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 1 truck, 176.1ms
Speed: 3.9ms preprocess, 176.1ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 1 truck, 179.7ms
Speed: 4.4ms preprocess, 179.7ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 3 trucks, 179.3ms
Speed: 4.4ms preprocess, 179.3ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x6

In [ ]:
# ── Saving per-frame detections as JSON ─────────────────────────────────

# JSON structure: list of frames, each frame is a list of detection dicts
json_payload = {
    "video_file": VIDEO_PATH.name,
    "model_variant": "small",
    "confidence_threshold": CONF_THRESHOLD,
    "frame_count": results["frame_count"],
    "fps_video": results["fps_video"],
    "frame_detections": results["all_detections"],
    "latencies_ms": results["latencies_ms"],
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(json_payload, f, indent=2)

size_kb = OUTPUT_JSON.stat().st_size / 1024
print(f"Saved detections JSON: {OUTPUT_JSON}")
print(f"  File size : {size_kb:.1f} KB")
print(f"  Frames    : {results['frame_count']}")

Saved detections JSON: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\data\processed\detections.json
  File size : 4504.2 KB
  Frames    : 1501


In [ ]:
# ── Inspecting the detection JSON structure ──────────────────────────────

with open(OUTPUT_JSON, "r") as f:
    loaded = json.load(f)

print(f"JSON keys: {list(loaded.keys())}")
print(f"\nVideo metadata:")
print(f"  video_file        : {loaded['video_file']}")
print(f"  model_variant     : {loaded['model_variant']}")
print(f"  confidence        : {loaded['confidence_threshold']}")
print(f"  frame_count       : {loaded['frame_count']}")
print(f"  fps_video         : {loaded['fps_video']:.1f}")

# Inspect first frame detections
first_frame_dets = loaded["frame_detections"][0]
print(f"\nFrame 0 — {len(first_frame_dets)} detections:")
for det in first_frame_dets[:3]:
    print(f"  {det['class_name']} (conf={det['confidence']:.3f})")

JSON keys: ['video_file', 'model_variant', 'confidence_threshold', 'frame_count', 'fps_video', 'frame_detections', 'latencies_ms']

Video metadata:
  video_file        : sample_traffic.mp4
  model_variant     : small
  confidence        : 0.25
  frame_count       : 1501
  fps_video         : 25.0

Frame 0 — 9 detections:
  car (conf=0.902)
  car (conf=0.902)
  car (conf=0.900)


In [ ]:
# ── FPS timeline plot ─────────────────────────────────────────────────

latencies = np.array(results["latencies_ms"])
fps_per_frame = 1000.0 / latencies  # Convert ms/frame → FPS
frame_numbers = np.arange(len(latencies))

# Rolling average (window=10 frames) to smooth out frame-to-frame jitter
window = min(10, len(fps_per_frame))
fps_rolling = pd.Series(fps_per_frame).rolling(window=window, center=True).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Raw FPS
axes[0].plot(frame_numbers, fps_per_frame, color="steelblue", alpha=0.5, lw=0.8, label="Per-frame FPS")
axes[0].plot(frame_numbers, fps_rolling, color="crimson", lw=2, label=f"Rolling mean ({window} frames)")
axes[0].axhline(fps_per_frame.mean(), color="darkgreen", lw=1.5, ls="--",
                label=f"Mean FPS = {fps_per_frame.mean():.1f}")
axes[0].set_ylabel("Frames per second", fontsize=11)
axes[0].set_title("Inference Speed Over Video — YOLOv8-small", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.4)

# Latency in ms
axes[1].plot(frame_numbers, latencies, color="steelblue", alpha=0.5, lw=0.8)
axes[1].axhline(latencies.mean(), color="darkgreen", lw=1.5, ls="--",
                label=f"Mean = {latencies.mean():.1f} ms")
axes[1].set_xlabel("Frame number", fontsize=11)
axes[1].set_ylabel("Latency (ms/frame)", fontsize=11)
axes[1].set_title("Per-Frame Inference Latency", fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.4)

plt.tight_layout()
save_figure(fig, "04_frames_per_second_timeline")
plt.show()

# Summary statistics
print("\nLatency summary (ms/frame):")
print(f"  Mean   : {latencies.mean():.1f}")
print(f"  Median : {np.median(latencies):.1f}")
print(f"  Std    : {latencies.std():.1f}")
print(f"  p95    : {np.percentile(latencies, 95):.1f}")
print(f"  p99    : {np.percentile(latencies, 99):.1f}")
print(f"\nMean FPS : {fps_per_frame.mean():.1f}")

  Saved: reports\figures\04_frames_per_second_timeline.png
  Saved: reports\figures\04_frames_per_second_timeline.pdf
  Saved: paper\figures\04_frames_per_second_timeline.png
  Saved: paper\figures\04_frames_per_second_timeline.pdf

Latency summary (ms/frame):
  Mean   : 222.0
  Median : 201.2
  Std    : 120.8
  p95    : 313.7
  p99    : 464.7

Mean FPS : 4.8


In [ ]:
# ── Verification that the output video was created ───────────────────────────────────
if OUTPUT_VIDEO.exists():
    size_mb = OUTPUT_VIDEO.stat().st_size / (1024 ** 2)
    print(f"✓ Output video created: {OUTPUT_VIDEO}")
    print(f"  File size: {size_mb:.1f} MB")
else:
    print("WARNING: Output video was not created. Check for errors above.")

✓ Output video created: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\data\processed\annotated_output.mp4
  File size: 73.9 MB
